# SFT — Instruction tuning multi-turn (Section 6.2)

**Chỉ chạy sau khi Bước 1 (CPT) đã xong** — tức repo `{HF_USER}/Qwen3-1.7B-vi-cpt` (bản merged) đã tồn tại trên Hub. Notebook này dạy model CPT format hội thoại trợ lý trên `5CD-AI/Vietnamese-Multi-turn-Chat-Alpaca` (12.7k hội thoại multi-turn tiếng Việt).

Cùng cơ chế với `CPT-Step-B-Train-Qwen3-1.7B.ipynb`: **Run All mỗi session** — tự kéo checkpoint mới nhất từ Hub → train tiếp → push checkpoint đầy đủ mỗi `save_steps` → tự dừng trước khi hết giờ. Ước tính ~800 step (2 epoch, effective batch 32) ≈ **1 session T4 là xong**, nhưng resume vẫn có sẵn phòng session bị cắt.

Repo trên Hub:
- `{HF_USER}/Qwen3-1.7B-vi-cpt` — input (output Bước 1, đã merge)
- `{HF_USER}/Qwen3-1.7B-vi-sft-ckpt` — checkpoint để resume
- `{HF_USER}/Qwen3-1.7B-vi-sft` — bản merged cuối, input cho Bước 3 (Reward Model)

**Trước khi chạy:** bật GPU T4 x1; secret `HF_TOKEN` (**Write** token — cell Config tự kiểm tra). Username HF tự lấy từ token, không cần sửa gì trong notebook.

In [ ]:
!pip install -q -U unsloth

In [ ]:
# Config + login — username TỰ LẤY từ token (whoami), không gõ tay để khỏi sai namespace
import os, time
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi, login, snapshot_download, whoami
login(UserSecretsClient().get_secret("HF_TOKEN"))

info = whoami()
HF_USER = info["name"]
role = ((info.get("auth") or {}).get("accessToken") or {}).get("role")
assert role != "read", "HF_TOKEN là READ token — tạo token WRITE tại hf.co/settings/tokens rồi cập nhật Kaggle Secret"
print(f"HF user: {HF_USER} (token: {role})")

CPT_MODEL  = f"{HF_USER}/Qwen3-1.7B-vi-cpt"       # output Bước 1 (đã merge) — KHÔNG phải Qwen3 gốc
CKPT_REPO  = f"{HF_USER}/Qwen3-1.7B-vi-sft-ckpt"  # checkpoint resume
FINAL_REPO = f"{HF_USER}/Qwen3-1.7B-vi-sft"       # bản merged cuối — input cho Bước 3 (RM)
OUT_DIR    = "/kaggle/working/sft-checkpoints"
MAX_SEQ    = 2048
EPOCHS     = 2       # 12.7k hội thoại, effective batch 32 → ~800 step
BUDGET_H   = 8.0     # tự dừng sau 8h để kịp push trước khi session bị cắt (~9h)
SYSTEM_PROMPT = "Bạn là một trợ lý AI thân thiện, hãy trả lời bằng tiếng Việt."

In [ ]:
# Kéo checkpoint mới nhất từ Hub về (None nếu là session đầu tiên)
api = HfApi()
api.create_repo(CKPT_REPO, private=True, exist_ok=True)
steps = sorted(int(f.split("/")[0].split("-")[1]) for f in api.list_repo_files(CKPT_REPO)
               if f.startswith("checkpoint-"))
last_ckpt = None
if steps:
    name = f"checkpoint-{steps[-1]}"
    snapshot_download(CKPT_REPO, allow_patterns=f"{name}/*", local_dir=OUT_DIR)
    last_ckpt = os.path.join(OUT_DIR, name)
print("Resume từ:", last_ckpt or "(bắt đầu mới)")

In [ ]:
# Load model CPT đã merge (4-bit) + gắn adapter QLoRA r=16 — cấu hình cố định của Bước 2 (bảng 6.2)
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    CPT_MODEL, max_seq_length=MAX_SEQ, load_in_4bit=True, dtype=None)
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=16, lora_dropout=0,
    target_modules=["q_proj","k_proj","v_proj","up_proj","down_proj","o_proj","gate_proj"],
    use_rslora=True, use_gradient_checkpointing="unsloth", random_state=42)

In [ ]:
# Data: 5CD-AI/Vietnamese-Multi-turn-Chat-Alpaca — dạng ShareGPT ({"from": human/gpt, "value": ...})
# standardize_sharegpt → {"role", "content"}, rồi render thành ChatML với system prompt cố định.
# Dùng template "chatml" thuần thay template Qwen3 gốc (template gốc chèn block <think> — không cần
# cho SFT thường; token <|im_start|>/<|im_end|> vốn có sẵn trong vocab Qwen nên không phải thêm token)
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template, standardize_sharegpt

tokenizer = get_chat_template(tokenizer, chat_template="chatml")

ds = standardize_sharegpt(load_dataset("5CD-AI/Vietnamese-Multi-turn-Chat-Alpaca", split="train"))
def to_text(ex):
    msgs = [{"role": "system", "content": SYSTEM_PROMPT}] + ex["conversations"]
    return {"text": tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)}
ds = ds.map(to_text, remove_columns=ds.column_names).train_test_split(test_size=200, seed=42)
print(ds)
print(ds["train"][0]["text"][:600])

In [ ]:
# Callback: mỗi lần save, upload NGUYÊN thư mục checkpoint lên Hub (cả optimizer/scheduler/rng —
# session sau mới resume đúng) và tự dừng khi vượt ngân sách giờ. Giống hệt CPT-Step-B-Train-Qwen3-1.7B.ipynb.
from transformers import TrainerCallback
class KaggleSessionCallback(TrainerCallback):
    def __init__(self): self.t0 = time.time()
    def on_save(self, args, state, control, **kw):
        name = f"checkpoint-{state.global_step}"
        api.upload_folder(repo_id=CKPT_REPO, folder_path=os.path.join(args.output_dir, name),
                          path_in_repo=name)
        print(f"Đã push {name} lên {CKPT_REPO}")
        if time.time() - self.t0 > BUDGET_H * 3600:
            control.should_training_stop = True
            print("Hết ngân sách giờ session — dừng an toàn, session sau tự resume.")

In [ ]:
# SFTTrainer + train_on_responses_only: chỉ tính loss trên lượt assistant (mask câu hỏi của user)
# Nếu TRL bản mới báo lỗi tham số: chuyển dataset_text_field/max_seq_length sang trl.SFTConfig
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer,
    train_dataset=ds["train"], eval_dataset=ds["test"],
    dataset_text_field="text", max_seq_length=MAX_SEQ, packing=False,
    callbacks=[KaggleSessionCallback()],
    args=TrainingArguments(
        output_dir=OUT_DIR,
        per_device_train_batch_size=4, gradient_accumulation_steps=8,  # effective 32 như script gốc
        learning_rate=1e-4, lr_scheduler_type="cosine", warmup_ratio=0.05,
        num_train_epochs=EPOCHS,
        bf16=True, optim="paged_adamw_8bit",
        save_strategy="steps", save_steps=100, save_total_limit=2,
        eval_strategy="steps", eval_steps=200,
        logging_steps=10, report_to="none",
    ),
)
trainer = train_on_responses_only(trainer,
    instruction_part="<|im_start|>user\n", response_part="<|im_start|>assistant\n")
trainer.train(resume_from_checkpoint=last_ckpt)
print(f"global_step = {trainer.state.global_step} / {trainer.state.max_steps}")

In [ ]:
# Khi train đủ số step: merge LoRA vào model CPT, push bản final — Bước 3 (RM) load thẳng repo này
if trainer.state.global_step >= trainer.state.max_steps:
    model.push_to_hub_merged(FINAL_REPO, tokenizer, save_method="merged_16bit")
    print(f"SFT HOÀN TẤT — bản merged tại {FINAL_REPO}. Cập nhật README (bước 2 → done).")
else:
    print("Chưa xong — mở lại notebook này ở session sau, nó tự resume.")

In [ ]:
# Smoke test: model giờ phải trả lời theo format trợ lý (so tiêu chí 6.9 trong spec)
FastLanguageModel.for_inference(model)
msgs = [{"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": "Xin chào! Bạn có thể giới thiệu về Việt Nam trong 3 câu không?"}]
ids = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to(model.device)
out = model.generate(ids, max_new_tokens=200, do_sample=True, temperature=0.7)
print(tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True))